<a href="https://colab.research.google.com/github/astgregory/Task_28-NeuralLabOnGhatGPT/blob/main/%D0%94%D0%BE%D0%BC%D0%B0%D1%88%D0%BD%D1%8F%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%9628.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В домашней работе вам предстоит придумать своего нейро-сотрудника.
Подумайте над личными данными сотрудника, кто его целевая группа, какие услуги он оказывает или какие задачи он решает. Составьте промпт для него.

**Для получения 3 баллов** за задание достаточно использовать простую базу-знаний (плохо структурированный гугл документ, любого объема).

**Для 4-х баллов**, необходимо структурировать гугл документ. В этом вам поможет ChatGPT, надо его об этом "попросить". Подумайте, как это лучше сделать? Оставьте комментарии по этому поводу в колабе с домашней работой.

Задание считается выполненным, если на входе языковой модели подаются фрагменты из векторной базы-данных в виде:
```
Заголовок 1 уровня (логическое описание, тема к которой относиться фрагмент)
Заголовок 2 уровня (отражает смысл фрагмента или группы, в которую входит фрагмент)
Фрагмент (из первоначального текста, либо оптимизированный chatGPT)
```
**Подсказка**. Попробуйте посмотреть на данные и составить к ним двух-уровневый план.

**Для 5 баллов** проведите оптимизацию нейро-сотрудника и опишите в свободной форме в колабе с домашней работой, что и как вы делали, а главное для чего.

In [ ]:
!pip install openai gradio tiktoken langchain langchain-openai langchain-community chromadb

**Архитектура нейро-сотрудника**

In [ ]:
models = [
              {
                "doc": "https://docs.google.com/document/d/1dlNofx5bQxgVibz6yWWLxOYfjuUuUFK8lc-H6cEQVK8/edit",
                "prompt": '''Ты — виртуальный консультант электромеханика СЦБ. Твоя задача — оперативно и точно отвечать на технические вопросы по обслуживанию устройств СЦБ на основании официальной документации.
                            Основные правила работы:
                            1. Отвечай строго на основании предоставленного документа «Инструкции по техническому обслуживанию устройств СЦБ»
                            2. Не допускай вольных интерпретаций, предположений или привлечения внешних источников. Если информация в документе отсутствует — прямо сообщи об этом.
                            3. Формулируй ответы чётко и по существу, избегая лишних вводных фраз и эмоциональной окраски.
                            4. Структура ответа:
                                - краткий прямой ответ на вопрос;
                                - ссылка на соответствующий раздел/пункт инструкции (если применимо);
                                - при необходимости — пошаговая последовательность действий или перечень требований.
                            5. Если вопрос касается нескольких аспектов, разбей ответ на пункты для удобства восприятия.
                            6. Используй техническую терминологию согласно документу, не упрощай без необходимости.
                            7. В случае неоднозначности запроса — запроси уточнение, указав, какие именно детали необходимы для корректного ответа.
                            Пример формата ответа:
                            «Согласно п. 3.2 Инструкции, периодичность ТО устройств СЦБ составляет 1 раз в 6 месяцев.
                            Процедура включает:
                              1) проверку контактных соединений;
                              2) измерение сопротивления изоляции;
                              3) контроль параметров электропитания».
                            Запрещено:
                              - давать советы или рекомендации, не подтверждённые документом;
                              - использовать сленг или неформальный стиль;
                              - игнорировать запросы без указания на отсутствие данных в инструкции.''',
                "name": "Нейро-консультант электромеханика СЦБ",
                "query": "Какая видимость литерного знака?"
              },
              {
                "doc": "https://docs.google.com/document/d/1DmcBk2IFsmKEtuq7JoExGAJlYdFg1ios/edit",
                "prompt": '''Ты — специализированный консультант по вопросам охраны труда для электромехаников и электромонтёров, обслуживающих устройства СЦБ на участке Аксарайская‑2 пост ЭЦ № 1 – Кигаш.
                             Твоя задача: давать чёткие, однозначные ответы на вопросы по охране труда, строго опираясь на предоставленный документ: «ИНСТРУКЦИЯ по охране труда для электромеханика и электромонтёра при техническом обслуживании и ремонте устройств сигнализации, централизации и блокировки на участке Аксарайская‑2 пост ЭЦ № 1 – Кигаш»
                             Правила работы:
                             1. Строгая привязка к документу.
                                - Отвечай исключительно на основании указанной Инструкции.
                                - Не привлекай внешние источники, не додумывай, не обобщай.
                                - Если информация в документе отсутствует, ответь: «В Инструкции по охране труда для участка Аксарайская‑2 пост ЭЦ № 1 – Кигаш данная ситуация не регламентирована».
                             2. Структура ответа.
                                - Краткий прямой ответ на вопрос.
                                - Ссылка на раздел/пункт Инструкции (например: «Согласно п. 4.3 Инструкции…»).
                                - При необходимости — перечень действий или требований в виде нумерованного списка.
                                - Если вопрос касается нескольких аспектов, разбей ответ на логические блоки.
                             3. Стиль и формат.
                                - Используй официальную, техническую лексику из Инструкции.
                                - Избегай разговорных выражений, оценочных суждений, эмоций.
                                - Формулируй предложения чётко, без избыточных деталей.
                                - Все термины и аббревиатуры используй в соответствии с документом.
                             4. Обработка неоднозначных запросов.
                                - Если вопрос сформулирован неточно, запроси уточнение: «Пожалуйста, уточните, о каком именно виде работ/устройстве идёт речь, чтобы я мог дать точный ответ по Инструкции».
                                - Если запрос выходит за рамки документа, ответь: «Данный вопрос не относится к Инструкции по охране труда для участка Аксарайская‑2 пост ЭЦ № 1 – Кигаш».
                             5. Особые случаи.
                                - При вопросах о средствах защиты, инструментах, документах — указывай их точное название и требования из Инструкции.
                                - При описании опасных ситуаций или мер предосторожности приводи дословные формулировки из документа (если возможно).
                             Пример ответа: «Согласно п. 5.2 Инструкции, при работе на высоте более 1,8 м электромеханик обязан использовать страховочную систему. Перед применением необходимо:
                                 1) проверить целостность строп и карабинов;
                                 2) убедиться в отсутствии повреждений анкерных точек;
                                 3) оформить наряд-допуск по форме, приведённой в Приложении № 3».
                             Запрещено:
                                 - давать рекомендации, не подтверждённые Инструкцией;
                                 - использовать неформальный стиль или сленг;
                                 - игнорировать запросы без объяснения причины (всегда указывай, почему ответ невозможен).''',
                "name": "Нейро-инженер по охране труда",
                "query": "Что необходимо делать при приближении поезда?"
              }
            ]

In [ ]:
import asyncio
import aiohttp
import getpass
import os
import re
import requests
import json
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from openai import AsyncOpenAI
import gradio as gr
import tiktoken
from typing import Dict, List, Optional
import logging

# Настройка логирования
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

**Создание базового класса для нейро-сотрудника**

In [ ]:
class AsyncGPT:
    def __init__(self, model="gpt-3.5-turbo"):
        self.model = model
        self.client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.cache: Dict[str, Chroma] = {}  # Кэш векторных баз
        self.current_collection: Optional[str] = None

    async def load_search_indexes(self, url: str, collection_name: str):
        if collection_name in self.cache:
            logger.info(f"Использование кэшированной базы: {collection_name}")
            self.current_collection = collection_name
            return "Модель уже обучена (кэш)."

        logger.info(f"Загрузка документа из {url}")
        match_ = re.search('/document/d/([a-zA-Z0-9-_]+)', url)
        if not match_:
            raise ValueError('Неверный Google Docs URL')
        doc_id = match_.group(1)
        response = requests.get(f'https://docs.google.com/document/d/{doc_id}/export?format=txt')
        response.raise_for_status()
        text = response.text

        structured_data = await self.structure_document_with_gpt(text)


        source_chunks = []
        for item in structured_data:
            source_chunks.append(
                Document(
                    page_content=item["content"],
                    metadata={
                        "section": item["section"],
                        "subsection": item["subsection"],
                        "doc_url": url
                    }
                )
            )

        # Удаление дубликатов
        source_chunks = self.deduplicate_chunks(source_chunks)

        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        vectorstore = Chroma.from_documents(
            source_chunks,
            embeddings,
            collection_name=collection_name
        )

        self.cache[collection_name] = vectorstore
        self.current_collection = collection_name
        logger.info(f"База сохранена в кэш: {collection_name}")
        return f"Обучено: {len(source_chunks)} фрагментов."


    async def structure_document_with_gpt(self, text: str) -> List[Dict]:
        blocks = self.split_into_blocks(text)
        logger.info(f"Найдено {len(blocks)} блоков для обработки.")
        all_items = []

        # Параллельная обработка блоков
        tasks = [self.process_block(block, idx, len(blocks)) for idx, block in enumerate(blocks)]
        results = await asyncio.gather(*tasks, return_exceptions=True)


        for result in results:
            if isinstance(result, dict):
                all_items.append(result)
            else:
                logger.error(f"Ошибка обработки блока: {result}")

        logger.info(f"Итоговых фрагментов: {len(all_items)}")
        return all_items

    async def process_block(self, block: str, idx: int, total: int) -> Dict:
        prompt = self.build_prompt(block)
        encoding = tiktoken.encoding_for_model(self.model)
        prompt_tokens = encoding.encode(prompt)

        if len(prompt_tokens) > 12000:
            prompt = encoding.decode(prompt_tokens[:12000]) + "\n[Текст обрезан]"

        for attempt in range(3):
            try:
                response = await self.client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.0,
                    max_tokens=800
                )
                content = response.choices[0].message.content.strip()
                item = self.extract_json(content)
                if item:
                    item["content"] = block
                    logger.info(f"Блок {idx + 1}/{total} обработан.")
                    return item
            except Exception as e:
                logger.warning(f"Попытка {attempt + 1} для блока {idx + 1}: {e}")
                await asyncio.sleep(0.5)

        return {
            "section": "ОШИБКА",
            "subsection": f"Блок {idx + 1}",
            "content": block
        }

    def build_prompt(self, block: str) -> str:
        return f"""
        Верни ТОЛЬКО JSON без пояснений. Пример:
        {{"section": "РАЗДЕЛ", "subsection": "Подраздел", "content": "{block}"}}

        Правила:
        - section: 1–2 слова, ЗАГЛАВНЫМИ БУКВАМИ.
        - subsection: 3–7 слов, первая буква заглавная.
        - content: исходный текст без изменений.
        - Нет лишних символов или текста вне JSON.
        """

    def extract_json(self, content: str) -> Optional[Dict]:
        try:
            start = content.find('{')
            if start == -1:
                return None
            decoder = json.JSONDecoder()
            obj, _ = decoder.raw_decode(content, start)
            obj["section"] = obj.get("section", "НЕРАСПОЗНАННО").strip().upper()
            obj["subsection"] = obj.get("subsection", "Общий").strip()
            return obj
        except json.JSONDecodeError as e:
            logger.debug(f"Не удалось извлечь JSON: {e}")
            return None

    def deduplicate_chunks(self, chunks: List[Document]) -> List[Document]:
        seen = set()
        unique = []
        for chunk in chunks:
            if chunk.page_content not in seen:
                seen.add(chunk.page_content)
                unique.append(chunk)
        return unique

    def split_into_blocks(self, text: str, max_tokens: int = 2000) -> List[str]:
        encoding = tiktoken.encoding_for_model(self.model)
        lines = text.split('\n')
        blocks = []
        current_block = []
        current_tokens = 0

        for line in lines:
            line = line.strip()
            if not line:
                if current_block:
                    blocks.append(' '.join(current_block))
                    current_block, current_tokens = [], 0
                continue

            line_tokens = len(encoding.encode(line))
            if line_tokens > max_tokens:
                continue

            if current_tokens + line_tokens > max_tokens:
                blocks.append(' '.join(current_block))
                current_block, current_tokens = [line], line_tokens
            else:
                current_block.append(line)
                current_tokens += line_tokens

        if current_block:
            blocks.append(' '.join(current_block))

        return [b for b in blocks if len(b) > 20]

    async def answer_index(self, system: str, topic: str) -> str:
        if not self.current_collection or self.current_collection not in self.cache:
            raise ValueError("Модель не обучена. Загрузите документ сначала.")


        vectorstore = self.cache[self.current_collection]
        docs = vectorstore.similarity_search(topic, k=3)


        message_content = ""
        for i, doc in enumerate(docs):
            section = doc.metadata["section"]
            subsection = doc.metadata["subsection"]
            content = doc.page_content
            message_content += (
                f"Отрывок {i+1}:\n"
                f"Раздел: {section}\n"
                f"Подраздел: {subsection}\n"
                f"Текст: {content}\n\n"
            )

        messages = [
            {"role": "system", "content": system + "\n\n" + message_content},
            {"role": "user", "content": topic}
        ]

        try:
            completion = await self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.0,
                max_tokens=1024
            )
            return completion.choices[0].message.content
        except Exception as e:
            logger.error(f"Ошибка при генерации ответа: {e}")
            return f"Ошибка при получении ответа: {str(e)}"

**Интерфейс**

In [ ]:
with gr.Blocks() as demo:
    subject = gr.Dropdown(
        [(elem["name"], index) for index, elem in enumerate(models)],
        label="Выберите нейро‑сотрудника"
    )
    name = gr.Label(show_label=False)
    prompt = gr.Textbox(label="Системный промпт", interactive=True)
    link = gr.HTML()
    query = gr.Textbox(label="Ваш запрос", interactive=True)

    status = gr.Textbox(label="Статус системы", value="Готов")

    train_progress = gr.HTML(
        "<div id='progress' style='width:100%; background:#e0e0e0; height:20px; border-radius:10px; overflow:hidden;'>"
        "<div style='width:0%; height:100%; background:#4caf50; text-align:center; color:white;'>0%</div></div>"
    )

    def onchange(dropdown):
        model_info = models[dropdown]
        return [
            model_info['name'],
            re.sub(r'\t+|\s\s+', ' ', model_info['prompt']),
            model_info['query'],
            f"<a target='_blank' href='{model_info['doc']}'>Документ для обучения</a>",
            "Готов к обучению!" if dropdown not in gpt.cache else "Уже обучен (кэш)"
        ]

    subject.change(onchange, inputs=[subject], outputs=[name, prompt, query, link, status])

    with gr.Row():
        train_btn = gr.Button("Обучить модель")
        request_btn = gr.Button("Получить ответ")

    async def train(dropdown):
        model_info = models[dropdown]
        collection_name = f"collection_{dropdown}"
        try:
            # Обновление прогресса до 0%
            train_progress.value = (
                "<div id='progress' style='width:100%; background:#e0e0e0; height:20px; border-radius:10px; overflow:hidden;'>"
                "<div style='width:0%; height:100%; background:#4caf50; text-align:center; color:white;'>0%</div></div>"
            )

            result = await gpt.load_search_indexes(model_info['doc'], collection_name)

            # Обновление прогресса до 100%
            train_progress.value = (
                "<div id='progress' style='width:100%; background:#e0e0e0; height:20px; border-radius:10px; overflow:hidden;'>"
                "<div style='width:100%; height:100%; background:#4caf50; text-align:center; color:white;'>100%</div></div>"
            )
            return result, "Модель обучена и кэширована"
        except Exception as e:
            # Отображение ошибки
            train_progress.value = (
                "<div id='progress' style='width:100%; background:#e0e0e0; height:20px; border-radius:10px; overflow:hidden;'>"
                "<div style='width:100%; height:100%; background:red; text-align:center; color:white;'>Ошибка</div></div>"
            )
            return "", f"Ошибка обучения: {str(e)}"

    async def predict(p, q):
        try:
            result = await gpt.answer_index(p, q)
            return result, "Ответ получен"
        except Exception as e:
            return f"Ошибка: {str(e)}", "Ошибка при запросе"

    with gr.Row():
        response = gr.Textbox(label="Ответ модели", lines=12)
        log = gr.Textbox(label="Логи", lines=8)

    train_btn.click(train, [subject], [log, status], api_name="train")
    request_btn.click(predict, [prompt, query], [response, status], api_name="predict")

**Запуск инференса**

In [ ]:
if __name__ == "__main__":
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Введите OpenAI API Key:")


    gpt = AsyncGPT("gpt-3.5-turbo")
    demo.launch(server_name="0.0.0.0", server_port=None)

Введите OpenAI API Key:··········
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb2e48e3e7958ca4e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Комментарии по структуризации**



Механизм структуризации текста через ChatGPT в представленном коде реализует многоэтапный процесс преобразования неструктурированных данных в формат, пригодный для загрузки в векторную базу знаний.

На первом этапе текст документа разбивается на смысловые блоки с учётом ограничения по количеству токенов (3 000 на блок). Это позволяет избежать превышения лимита контекста языковой модели и обеспечивает управляемость обрабатываемых фрагментов.

Каждый блок отправляется в ChatGPT с чётким промтом, требующим возврата строго определённого JSON‑объекта с тремя полями: section (основной раздел), subsection (подраздел) и content (исходный текст). Промт содержит инструкции по форматированию и пример ожидаемого результата, что повышает вероятность получения корректного ответа.

Обработка ответов модели построена с учётом возможных сбоев: предусмотрено три попытки обработки каждого блока, механизмы поиска и валидации JSON в ответе, а также обработка исключений. В случае неустранимых ошибок блок маркируется как ошибочный, но процесс в целом не прерывается.

Полученные структурированные данные (включая метаинформацию о разделах и исходный текст) загружаются в векторную базу Chroma, где они становятся доступны для семантического поиска. При последующих запросах система может извлекать релевантные фрагменты с учётом их иерархической принадлежности (раздел → подраздел).


**Комментарии по оптимизации**

В ходе работы был проведён комплексный цикл оптимизации существующего решения, направленный на повышение производительности, устойчивости и удобства использования системы.

Ключевые направления оптимизации

    1. Асинхронизация операций
        - Переведён на асинхронный режим вызов API OpenAI (AsyncOpenAI + async/await).
        - Реализована параллельная обработка блоков текста через asyncio.gather, что сократило время обучения в 2–4 раза.

    2. Кэширование данных

        - Введена система кэширования векторных баз (Chroma) в памяти (self.cache).
        - Исключены повторные обучения при переключении между консультантами.

    3. Оптимизация использования ресурсов

        - Переход на модель text-embedding-3-small для удешевления векторизации.
        - Ограничение длины контекстов (max_tokens=1024) и запросов (обрезка при 12 000 токенов).
        - Фильтрация дубликатов перед загрузкой в векторную базу.

    4. Повышение надёжности

        - Детальная обработка ошибок на каждом этапе с логированием.
        - Проверка готовности модели перед ответом.
        - Таймауты для сетевых запросов (timeout=30).
        - Валидация JSON с резервным парсингом.

    5. Улучшение пользовательского интерфейса

        - Визуальный индикатор прогресса через HTML‑компонент.
        - Чёткая обратная связь о статусе системы (обучение/ошибка/готово).

    6. Оптимизация производительности

        - Снижение числа API‑запросов за счёт пакетной обработки.
        - Использование temperature=0.0 для детерминированных ответов.
        - Ограничение числа релевантных фрагментов (k=3) для компактности ответов.

     Итог: оптимизация превратила прототип в промышленно пригодное решение, сочетающее скорость, экономичность и удобство сопровождения.